In [0]:
# ==========================================
# 03 - SILVER TRANSFORMATION
# Fraud Detection Lakehouse
# ==========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG_NAME = "fraud_detection"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
QUARANTINE_SCHEMA = "quarantine"

spark.sql(f"USE CATALOG {CATALOG_NAME}")

print("Silver transformation initialized")

In [0]:
# ------------------------------------------
# Duplicate detection
# ------------------------------------------

def split_duplicates(df, key_column):

    window = (
        Window
        .partitionBy(key_column)
        .orderBy(
            F.col("_ingestion_timestamp").asc(),
            F.col("_source_file").asc()
        )
    )

    ranked = df.withColumn(
        "_duplicate_rank",
        F.row_number().over(window)
    )

    valid = (
        ranked
        .filter(F.col("_duplicate_rank") == 1)
        .drop("_duplicate_rank")
    )

    duplicates = (
        ranked
        .filter(F.col("_duplicate_rank") > 1)
        .drop("_duplicate_rank")
        .withColumn(
            "_quarantine_reason",
            F.lit(f"Duplicate {key_column}")
        )
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    return valid, duplicates


# ------------------------------------------
# Data validation
# ------------------------------------------

def split_invalid_records(df, validation_rules):

    conditions = [
        F.when(condition, F.lit(reason))
        for condition, reason in validation_rules
    ]

    tagged = df.withColumn(
        "_quarantine_reason",
        F.concat_ws("; ", *conditions)
    )

    invalid = (
        tagged
        .filter(F.length("_quarantine_reason") > 0)
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    valid = (
        tagged
        .filter(F.length("_quarantine_reason") == 0)
        .drop("_quarantine_reason")
    )

    return valid, invalid

In [0]:
# ------------------------------------------
# Customer transformation
# ------------------------------------------

customers = spark.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.customers"
)

customers = (
    customers
    .withColumn(
        "age",
        F.expr("try_cast(age AS INT)")
    )
    .withColumn(
        "registration_date",
        F.to_date("registration_date")
    )
)

# Remove duplicate customer IDs
customers, duplicate_customers = split_duplicates(
    customers,
    "customer_id"
)

# Apply validation rules
customers, invalid_customers = split_invalid_records(
    customers,
    [
        (F.col("customer_id").isNull(), "Missing customer ID"),
        (
            F.col("age").isNull() |
            (F.col("age") < 18) |
            (F.col("age") > 100),
            "Invalid customer age"
        ),
        (
            F.col("registration_date").isNull(),
            "Invalid registration date"
        ),
        (
            F.col("country").isNull() |
            (F.trim("country") == ""),
            "Missing customer country"
        )
    ]
)

# Save valid customers
(
    customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{SILVER_SCHEMA}.customers")
)

# Save invalid and duplicate customers
customer_quarantine = duplicate_customers.unionByName(
    invalid_customers
)

(
    customer_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG_NAME}.{QUARANTINE_SCHEMA}.invalid_customers"
    )
)

print("Customer transformation completed")

In [0]:
# ------------------------------------------
# Merchant transformation
# ------------------------------------------

merchants = spark.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.merchants"
)

# Remove duplicate merchant IDs
merchants, duplicate_merchants = split_duplicates(
    merchants,
    "merchant_id"
)

# Apply validation rules
merchants, invalid_merchants = split_invalid_records(
    merchants,
    [
        (F.col("merchant_id").isNull(), "Missing merchant ID"),
        (
            F.col("category").isNull() |
            (F.trim("category") == ""),
            "Missing merchant category"
        ),
        (
            F.col("country").isNull() |
            (F.trim("country") == ""),
            "Missing merchant country"
        )
    ]
)

# Save valid merchants
(
    merchants.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{SILVER_SCHEMA}.merchants")
)

# Save quarantined merchants
merchant_quarantine = duplicate_merchants.unionByName(
    invalid_merchants
)

(
    merchant_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG_NAME}.{QUARANTINE_SCHEMA}.invalid_merchants"
    )
)

print("Merchant transformation completed")

In [0]:
# ------------------------------------------
# Transaction transformation
# ------------------------------------------

transactions = spark.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.transactions"
)

# Convert source data types
transactions = (
    transactions
    .withColumn(
        "amount",
        F.expr("try_cast(amount AS DECIMAL(18,2))")
    )
    .withColumn(
        "fraud_label",
        F.expr("try_cast(fraud_label AS INT)")
    )
    .withColumn(
        "transaction_timestamp",
        F.to_timestamp("transaction_timestamp")
    )
)

# Remove duplicate transaction IDs
transactions, duplicate_transactions = split_duplicates(
    transactions,
    "transaction_id"
)

# Get valid reference IDs
valid_customers = (
    spark.table(f"{CATALOG_NAME}.{SILVER_SCHEMA}.customers")
    .select("customer_id")
    .withColumn("_customer_exists", F.lit(True))
)

valid_merchants = (
    spark.table(f"{CATALOG_NAME}.{SILVER_SCHEMA}.merchants")
    .select("merchant_id")
    .withColumn("_merchant_exists", F.lit(True))
)

# Check referential integrity
transactions = (
    transactions
    .join(valid_customers, "customer_id", "left")
    .join(valid_merchants, "merchant_id", "left")
)

# Apply validation rules
transactions, invalid_transactions = split_invalid_records(
    transactions,
    [
        (
            F.col("transaction_id").isNull(),
            "Missing transaction ID"
        ),
        (
            F.col("_customer_exists").isNull(),
            "Customer reference not found"
        ),
        (
            F.col("_merchant_exists").isNull(),
            "Merchant reference not found"
        ),
        (
            F.col("amount").isNull() |
            (F.col("amount") <= 0),
            "Invalid transaction amount"
        ),
        (
            F.col("transaction_timestamp").isNull(),
            "Invalid transaction timestamp"
        ),
        (
            F.col("fraud_label").isNull() |
            (~F.col("fraud_label").isin(0, 1)),
            "Invalid fraud label"
        ),
        (
            F.col("status").isNull() |
            (F.trim("status") == ""),
            "Missing transaction status"
        ),
        (
            F.col("channel").isNull() |
            (F.trim("channel") == ""),
            "Missing transaction channel"
        )
    ]
)

# Remove temporary reference columns
transactions = transactions.drop(
    "_customer_exists",
    "_merchant_exists"
)

invalid_transactions = invalid_transactions.drop(
    "_customer_exists",
    "_merchant_exists"
)

# Save valid transactions
(
    transactions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG_NAME}.{SILVER_SCHEMA}.transactions"
    )
)

# Combine duplicate and invalid transactions
transaction_quarantine = duplicate_transactions.unionByName(
    invalid_transactions
)

# Save quarantined transactions
(
    transaction_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG_NAME}.{QUARANTINE_SCHEMA}.invalid_transactions"
    )
)

print("Transaction transformation completed")

In [0]:
# ------------------------------------------
# Silver and quarantine record counts
# ------------------------------------------

TABLES = [
    "silver.customers",
    "silver.merchants",
    "silver.transactions",
    "quarantine.invalid_customers",
    "quarantine.invalid_merchants",
    "quarantine.invalid_transactions"
]

for table in TABLES:

    full_table_name = f"{CATALOG_NAME}.{table}"

    count = spark.table(full_table_name).count()

    print(f"{full_table_name}: {count:,} records")